In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
import os
from groq import Groq

# Initialize client
client = Groq(
    api_key=os.environ.get("GROQ_API_KEY")
)

import import_ipynb
from baseline_gui import rag_kg_baseline, zero_shot_baseline
from fine_tuning_gui import generate_answer

Loaded 485 facts from KG
Loaded 134 text chunks
NVIDIA GeForce RTX 2060
VRAM: 6.44 GB


Loading checkpoint shards: 100%|██████████| 3/3 [00:35<00:00, 11.76s/it]


In [ ]:
import re
import gradio as gr

# define answers connecting backend functions
def zero_shot_answer(question):
    try:
        answer, metadata = zero_shot_baseline(question)
        sources = metadata.get("pages_cited", [])
    except Exception as e:
        answer = f"Error: {e}"
        sources = ["N/A"]
    return {"answer": answer, "sources": sources}


def rag_kg_answer(question):
    try:
        answer, metadata = rag_kg_baseline(question)
        sources = metadata.get("pages_cited", [])
    except Exception as e:
        answer = f"Error: {e}"
        sources = ["N/A"]
    return {"answer": answer, "sources": sources}


def fine_tuned_answer(question):
    try:
        answer, sources = generate_answer(question)
    except Exception as e:
        answer = f"Error: {e}"
        sources = ["N/A"]
    return {"answer": answer, "sources": sources}


# helper function to clean markdown
def clean_markdown(text: str) -> str:
    """
    Simplify Markdown-rich text for display in a Textbox.
    Removes headers, bold, tables, and symbols.
    """
    text = re.sub(r"(?m)^#{1,6}\s*", "", text)
    text = re.sub(r"\*\*(.*?)\*\*", r"\1", text)
    text = re.sub(r"\*(.*?)\*", r"\1", text)
    text = re.sub(r"`(.*?)`", r"\1", text)
    text = re.sub(r"\|", " | ", text)
    text = re.sub(r"---+", "", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()



# handler function
def answer_question(question, mode):
    if not question.strip():
        return "Please enter a question.", ""

    if mode == "Zero-shot":
        res = zero_shot_answer(question)
    elif mode == "RAG-KG":
        res = rag_kg_answer(question)
    else:
        res = fine_tuned_answer(question)

    sources_text = ", ".join(map(str, res["sources"])) if res.get("sources") else "N/A"

    # clean the answer for display
    clean_answer = clean_markdown(res["answer"])
    return clean_answer, sources_text





with gr.Blocks(css="""
    /* Header styling: purple-pink gradient */
    .header {
        background: linear-gradient(90deg, #D291BC, #957DAD);
        padding: 15px;
        border-radius: 12px;
        color: white;
        font-weight: bold;
        text-align: center;
        font-size: 24px;
    }

    /* Answer box: soft pastel pink background with purple border */
    .answer-box textarea {
        border: 2px solid #957DAD;
        border-radius: 12px;
        background-color: #F8E1F4;
        font-size: 14px;
        padding: 12px;
        color: #5E2D79;
    }

    /* Sources box: light pastel blue background with soft purple border */
    .source-box textarea {
        border: 1px solid #B39BC8;
        border-radius: 10px;
        background-color: #E3E4FF;
        font-size: 13px;
        padding: 8px;
        color: #4A3F8C;
    }

    /* Primary button: pink gradient with hover effect */
    .btn-primary {
        background: linear-gradient(90deg, #F48FB1, #F06292);
        color: white;
        font-weight: bold;
        border-radius: 12px;
        border: none;
        padding: 12px 25px;
        transition: background 0.3s;
    }
    .btn-primary:hover {
        background: linear-gradient(90deg, #F06292, #EC407A);
    }

    /* Row/column spacing */
    .gr-row, .gr-column {
        gap: 12px;
    }
""") as demo:

    # Header
    gr.Markdown("## 🥦 Sustainable Health from Food - Q&A 🌸", elem_classes="header")
    gr.Markdown(
        "Ask a question about nutrition, ingredients, or food sustainability. "
        "Switch between **Zero-shot**, **RAG-KG**, and **Fine-tuned** modes."
    )

    # Input + mode
    with gr.Row():
        with gr.Column(scale=2):
            question = gr.Textbox(
                label="Your Question",
                placeholder="e.g., What foods are rich in vitamin C?",
                lines=2
            )
            mode = gr.Radio(
                ["Zero-shot", "RAG-KG", "Fine-tuned"],
                value="RAG-KG",
                label="Mode"
            )
            submit = gr.Button("Get Answer", elem_classes="btn-primary")
        
        # Output area
        with gr.Column(scale=3):
            output = gr.Textbox(label="Answer", interactive=False, lines=10, elem_classes="answer-box")
            sources = gr.Textbox(label="Sources (pages/sections)", interactive=False, lines=2, elem_classes="source-box")

    # Note
    gr.Markdown("---")
    gr.Markdown("**Note:** Answers are grounded in your Knowledge Graph when using RAG-KG or Fine-tuned modes.")

    # Connect function
    submit.click(fn=answer_question, inputs=[question, mode], outputs=[output, sources])

# Run the app
if __name__ == "__main__":
    #demo.launch(share=True)
    demo.launch()


* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
